# Notebook 1 — Graph RAG on TechNova's Organisation (Neo4j + LangChain)
## Advanced RAG | Section 1: Graph RAG

**Author:** Suresh D R | AI Product Developer & Technology Mentor

---

We build a **company knowledge graph** for TechNova — the same people, teams, projects and systems used throughout the Advanced RAG document:

| Who / What | Role in the Story |
|---|---|
| **Arjun Mehta** | CEO |
| **Priya Sharma** | CTO — reports to Arjun |
| **Karan Bhatia** | Engineering Manager, leads the **Platform** team, owns **Project Orion** |
| **Divya Rao** | Head of **Data & AI** — Projects Atlas and Sage |
| **Rahul Verma, Sneha Iyer, Arvind Nair, …** | Engineers on the Platform team |
| **Payments API → Auth Service → Redis Cluster 2** | The dependency chain behind the September payments incident |

The graph is built from **two kinds of data** — exactly like in a real company:

| Kind | Source Files | How It Gets Into the Graph |
|---|---|---|
| **Structured** | `employees.csv`, `teams.csv`, `projects.csv`, `project_assignments.csv`, `services.csv` | Loaded directly with Cypher — IDs make it exact (no LLM needed) |
| **Unstructured** | Org update, Orion status report, incident report (PDF) | GPT-4o extracts entities and relationships with LangChain's `LLMGraphTransformer` |

**Ready-made tools used:** Neo4j · `Neo4jGraph` · `LLMGraphTransformer` · `add_graph_documents` · APOC · `Neo4jVector` · `GraphCypherQAChain` · `pymupdf4llm`

| Step | What We Do | Doc Section |
|---|---|---|
| 0 | Setup: OpenAI + Neo4j | — |
| 1 | Look at the data | — |
| 2 | Load the **structured** data into Neo4j | 1.2, 1.5 |
| 3 | Look at the graph | 1.2 |
| 4 | Load the **documents**: extract entities with GPT-4o and link them to the same graph | 1.6 |
| 5 | Entity resolution — "the CTO" is Priya Sharma | 1.7 |
| 6 | Vector index in Neo4j — and watch vector-only RAG fail | 1.1 |
| 7 | Real-world questions answered with Cypher | 1.5, 1.8 |
| 8 | Text-to-Cypher: ask in English | 1.8 |
| 9 | Hybrid Graph + Vector RAG | 1.3, 1.10 |

---
### Before You Start

**1. The data:** the folder `technova_data/` must be next to this notebook (upload/unzip `technova_data.zip` — the next cell does it for you).

**2. A Neo4j database (free, 5 minutes):**

| Option | How |
|---|---|
| **Neo4j AuraDB Free** (easiest) | console.neo4j.io → create a *Free* instance → save the password. APOC is included. |
| **Docker** | `docker run -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password123 -e NEO4J_PLUGINS='["apoc"]' neo4j:5` |

Needs Neo4j **5.26 or newer** with **APOC**. ⚠️ Use an **empty** database — Step 2 deletes everything in it.

**3. Setting up Neo4j AuraDB Free, step by step:**

| Step | What to Do |
|---|---|
| 1 | Go to **console.neo4j.io** → sign in → **Create instance** → **AuraDB Free** |
| 2 | Click **Download** on the credentials screen — the password is shown **only once** |
| 3 | Wait until the instance shows **RUNNING** (2–5 minutes) under **Instances** |

**4. What to paste when the Setup cell asks** (all values are in the downloaded credentials file):

| Prompt | Value | Example |
|---|---|---|
| OpenAI API key | From platform.openai.com → API keys | `sk-...` |
| Neo4j URI | `NEO4J_URI` | `neo4j+s://de249b8f.databases.neo4j.io` |
| Neo4j username | `NEO4J_USERNAME` — on new Aura instances this is the **instance ID**, not `neo4j` | `de249b8f` |
| Neo4j password | `NEO4J_PASSWORD` | (hidden) |

The **database name** (`NEO4J_DATABASE`) is also the instance ID on new Aura instances — the Setup cell sets it automatically from the username.

**Optional — Colab Secrets (no retyping):** click the 🔑 icon in Colab's left sidebar, add `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` (and `NEO4J_DATABASE`), turn on notebook access, and the Setup cell reads them automatically.

In [ ]:
%pip install -q openai langchain langchain-openai langchain-experimental langchain-neo4j neo4j pymupdf4llm pandas

---
## Get the Data (Google Colab or local)

This cell makes sure this notebook's data folder, **`technova_data/graph/`**, is available:

| Situation | What Happens |
|---|---|
| Folder already there | Nothing to do |
| `technova_data.zip` is next to the notebook | Extracts **only** `technova_data/graph/` from the zip |
| Running on **Google Colab** with no zip yet | Opens an **upload box** — choose `technova_data.zip` from your computer |

In [1]:
import zipfile
from pathlib import Path

NEEDED = "technova_data/graph/"                       # the only folder this notebook uses

if Path(NEEDED).exists() and any(Path(NEEDED).iterdir()):
    print("Data folder already available.")
else:
    zips = sorted(Path(".").glob("technova_data*.zip"))
    if zips:
        zip_path = zips[0]
    else:
        try:
            from google.colab import files               # only exists on Google Colab
        except ImportError:
            raise FileNotFoundError("Put technova_data.zip next to this notebook, then run this cell again.")
        print("Choose technova_data.zip from your computer:")
        uploaded = files.upload()
        zip_path = Path(next(name for name in uploaded if name.endswith(".zip")))
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist() if m.startswith(NEEDED)]
        if not members:
            raise ValueError(f"{zip_path.name} does not contain {NEEDED}")
        z.extractall(".", members=members)
    print(f"Extracted {len(members)} files from {zip_path.name} -> {NEEDED}")

for p in sorted(Path(NEEDED).iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size / 1024:>8.1f} KB")

Choose technova_data.zip from your computer:


Saving technova_data.zip to technova_data.zip
Extracted 9 files from technova_data.zip -> technova_data/graph/
  Engineering_Org_Update_Jul_2025.pdf           2.4 KB
  Incident_Report_INC-2025-0917.pdf             2.2 KB
  Project_Orion_Status_Q3_2025.pdf              2.8 KB
  employees.csv                                 2.5 KB
  project_assignments.csv                       0.5 KB
  projects.csv                                  0.5 KB
  services.csv                                  0.4 KB
  teams.csv                                     0.2 KB


---
## Step 0: Setup — OpenAI and Neo4j

Secrets are read from environment variables, or asked once in a hidden box. **Never type secrets into the notebook.**

In [4]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 19.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.3
    Uninstalling langchain-core-1.6.3:
      Successfully uninstalled langchain-core-1.6.3


In [6]:
!pip install langchain_neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.7/297.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 24.7 MB/s eta 0:00:00


In [7]:
import os, re, getpass, warnings
from pathlib import Path
import pandas as pd
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_neo4j import Neo4jGraph

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

try:                                                    # Google Colab Secrets (optional)
    from google.colab import userdata
    for v in ["OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE"]:
        try:
            os.environ.setdefault(v, userdata.get(v))
        except Exception:
            pass
except ImportError:
    pass

for var, prompt in [("OPENAI_API_KEY", "OpenAI API key"),
                    ("NEO4J_URI", "Neo4j URI (e.g. neo4j+s://xxxx.databases.neo4j.io or bolt://localhost:7687)"),
                    ("NEO4J_USERNAME", "Neo4j username (NEO4J_USERNAME in the credentials file)"),
                    ("NEO4J_PASSWORD", "Neo4j password")]:
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{prompt}: ") if ("KEY" in var or "PASSWORD" in var) else input(f"{prompt}: ")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# Database name: newer Aura instances name the database after the instance ID (e.g. "de249b8f"), not "neo4j".
# It is the NEO4J_DATABASE line in your Aura credentials file.
if not os.environ.get("NEO4J_DATABASE"):
    user = os.environ["NEO4J_USERNAME"]
    os.environ["NEO4J_DATABASE"] = user if user != "neo4j" else "neo4j"
NEO4J = dict(url=os.environ["NEO4J_URI"], username=os.environ["NEO4J_USERNAME"],
             password=os.environ["NEO4J_PASSWORD"], database=os.environ["NEO4J_DATABASE"])
graph = Neo4jGraph(**NEO4J, refresh_schema=False)

version = graph.query("CALL dbms.components() YIELD versions RETURN versions[0] AS v")[0]["v"]
print(f"Connected to Neo4j {version} | database: {os.environ['NEO4J_DATABASE']}")

DATA = Path("technova_data/graph")
assert DATA.exists(), "Data not found - run Notebook 0 first, or unzip technova_data.zip next to this notebook."

def show(title, text=""):
    print(f"\n=== {title} ===")
    if text:
        print(text)

OpenAI API key: ··········
Neo4j URI (e.g. neo4j+s://xxxx.databases.neo4j.io or bolt://localhost:7687): neo4j+s://de249b8f.databases.neo4j.io
Neo4j username (NEO4J_USERNAME in the credentials file): de249b8f
Neo4j password: ··········
Connected to Neo4j 5.27-aura | database: de249b8f


---
## Step 1: Look at the Data

Five CSV files from the HR and project-management systems, and three documents people wrote.

In [8]:
employees = pd.read_csv(DATA / "employees.csv", dtype=str).fillna("")
teams = pd.read_csv(DATA / "teams.csv", dtype=str)
projects = pd.read_csv(DATA / "projects.csv", dtype=str).fillna("")
assignments = pd.read_csv(DATA / "project_assignments.csv", dtype={"allocation_pct": int})
services = pd.read_csv(DATA / "services.csv", dtype=str).fillna("")

show(f"employees.csv - {len(employees)} people")
display(employees)
for name, df in [("teams.csv", teams), ("projects.csv", projects), ("services.csv", services)]:
    show(name)
    display(df)
show(f"project_assignments.csv - {len(assignments)} rows (first 6)")
display(assignments.head(6))
show("Documents", "\n".join(p.name for p in sorted(DATA.glob("*.pdf"))))


=== employees.csv - 24 people ===


,emp_id,name,title,level,department,team,manager_id,location,skills,joined_on
0,EMP001,Arjun Mehta,Chief Executive Officer,L10,Leadership,Leadership,,Bengaluru,Strategy;Fundraising,2016-04-01
1,EMP002,Priya Sharma,Chief Technology Officer,L9,Engineering,Leadership,EMP001,Bengaluru,Architecture;Kubernetes;Go,2019-06-10
2,EMP003,Neha Kapoor,Chief Human Resources Officer,L9,HR,Leadership,EMP001,Bengaluru,Talent;Compensation,2018-01-15
3,EMP004,Vikram Iyer,Chief Financial Officer,L9,Finance,Leadership,EMP001,Bengaluru,Finance;Compliance,2017-09-01
4,EMP005,Ravi Menon,VP Sales,L8,Sales,Leadership,EMP001,Pune,Enterprise Sales,2018-11-05
5,EMP010,Karan Bhatia,Engineering Manager,L5,Engineering,Platform,EMP002,Bengaluru,Go;Kubernetes;Python,2022-03-14
6,EMP011,Divya Rao,Head of Data & AI,L6,Data & AI,Data & AI,EMP002,Pune,Python;Spark;MLOps,2020-08-03
7,EMP012,Anil Kumar,Engineering Manager,L5,Engineering,Mobile,EMP002,Bengaluru,Kotlin;Swift,2021-05-17
8,EMP020,Arvind Nair,Lead Engineer,L4,Engineering,Platform,EMP010,Bengaluru,Go;Kubernetes;Redis,2021-02-01
9,EMP021,Sneha Iyer,Senior Engineer,L3,Engineering,Platform,EMP010,Bengaluru,Python;Go;PostgreSQL,2022-07-11



=== teams.csv ===


,team,department,lead_emp_id
0,Leadership,Leadership,EMP001
1,Platform,Engineering,EMP010
2,Mobile,Engineering,EMP012
3,Data & AI,Data & AI,EMP011
4,HR,HR,EMP003
5,Finance,Finance,EMP004
6,Sales,Sales,EMP005



=== projects.csv ===


...


=== services.csv ===


,service,owner_team,built_by_project,depends_on
0,Payments API,Platform,Orion,Auth Service;Postgres Main
1,Auth Service,Platform,,Redis Cluster 2;Postgres Main
2,Orders API,Platform,,Payments API
3,Mobile Backend,Mobile,Helios,Orders API;Auth Service
4,Analytics Pipeline,Data & AI,Atlas,Postgres Main
5,HRConnect Assistant,Data & AI,Sage,Auth Service
6,Redis Cluster 2,Platform,,
7,Postgres Main,Platform,,



=== project_assignments.csv - 17 rows (first 6) ===


,emp_id,project_id,role,allocation_pct
0,EMP010,P01,Project Lead,50
1,EMP020,P01,Tech Lead,100
2,EMP021,P01,Developer,100
3,EMP022,P01,Developer,100
4,EMP024,P01,Developer,60
5,EMP023,P01,DevOps,60



=== Documents ===
Engineering_Org_Update_Jul_2025.pdf
Incident_Report_INC-2025-0917.pdf
Project_Orion_Status_Q3_2025.pdf


---
## Step 2: Load the Structured Data Into Neo4j

Structured data has **IDs** (`EMP010`, `P01`), so we can load it **exactly** — no LLM, no guessing. This is the most reliable kind of entity resolution (Section 1.7: *"resolve on the ID first"*).

| From | Nodes Created | Relationships Created |
|---|---|---|
| `employees.csv` | `Person` (emp_id, title, level, joined_on), `Team`, `Location`, `Technology` | `REPORTS_TO`, `MEMBER_OF`, `LOCATED_IN`, `HAS_SKILL` |
| `teams.csv` | — | `LEADS` (team lead → team) |
| `projects.csv` | `Project` (status, dates) | `OWNS` (team → project), `USES` (project → technology) |
| `project_assignments.csv` | — | `WORKS_ON` {role, allocation_pct} |
| `services.csv` | `Service` | `MAINTAINS` (team → service), `BUILDS` (project → service), `DEPENDS_ON` |

Every node also gets the label **`__Entity__`** and an **`id`** = its name. `LLMGraphTransformer` uses the same convention in Step 4, so a person mentioned in a document **lands on the same node** as the CSV row.

Each load is one Cypher query with `UNWIND $rows` — Neo4j's standard way to bulk-load a list.

In [9]:
# ⚠️ Start from an empty database
graph.query("MATCH (n) DETACH DELETE n")
for idx in ["org_docs"]:
    graph.query(f"DROP INDEX {idx} IF EXISTS")
graph.query("CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (e:__Entity__) REQUIRE e.id IS UNIQUE")

emp_rows = employees.assign(skills=employees.skills.str.split(";")).to_dict("records")

graph.query("""
UNWIND $rows AS r
MERGE (p:__Entity__ {id: r.name}) SET p:Person,
      p.emp_id = r.emp_id, p.title = r.title, p.level = r.level, p.joined_on = r.joined_on
MERGE (t:__Entity__ {id: r.team}) SET t:Team
MERGE (p)-[:MEMBER_OF]->(t)
MERGE (l:__Entity__ {id: r.location}) SET l:Location
MERGE (p)-[:LOCATED_IN]->(l)
WITH p, r
UNWIND r.skills AS skill
MERGE (s:__Entity__ {id: skill}) SET s:Technology
MERGE (p)-[:HAS_SKILL]->(s)""", {"rows": emp_rows})

graph.query("""
UNWIND $rows AS r
MATCH (e:Person {emp_id: r.emp_id}), (m:Person {emp_id: r.manager_id})
MERGE (e)-[:REPORTS_TO]->(m)""", {"rows": employees[employees.manager_id != ""].to_dict("records")})

graph.query("""
UNWIND $rows AS r
MATCH (p:Person {emp_id: r.lead_emp_id}), (t:Team {id: r.team})
SET t.department = r.department
MERGE (p)-[:LEADS]->(t)""", {"rows": teams.to_dict("records")})

graph.query("""
UNWIND $rows AS r
MERGE (pr:__Entity__ {id: r.name}) SET pr:Project, pr.project_id = r.project_id, pr.status = r.status,
      pr.start_date = r.start_date, pr.target_date = r.target_date, pr.description = r.description
WITH pr, r
MATCH (t:Team {id: r.owner_team}) MERGE (t)-[:OWNS]->(pr)
WITH pr, r
UNWIND r.tech AS tech
MERGE (s:__Entity__ {id: tech}) SET s:Technology
MERGE (pr)-[:USES]->(s)""", {"rows": projects.assign(tech=projects.tech_stack.str.split(";")).to_dict("records")})

graph.query("""
UNWIND $rows AS r
MATCH (p:Person {emp_id: r.emp_id}), (pr:Project {project_id: r.project_id})
MERGE (p)-[w:WORKS_ON]->(pr) SET w.role = r.role, w.allocation_pct = r.allocation_pct""",
            {"rows": assignments.to_dict("records")})

svc_rows = services.assign(deps=services.depends_on.str.split(";").apply(lambda d: [x for x in d if x])).to_dict("records")
graph.query("""
UNWIND $rows AS r
MERGE (s:__Entity__ {id: r.service}) SET s:Service
WITH s, r
MATCH (t:Team {id: r.owner_team}) MERGE (t)-[:MAINTAINS]->(s)
WITH s, r
OPTIONAL MATCH (pr:Project {id: r.built_by_project})
FOREACH (_ IN CASE WHEN pr IS NULL THEN [] ELSE [1] END | MERGE (pr)-[:BUILDS]->(s))
WITH s, r
UNWIND r.deps AS dep
MERGE (d:__Entity__ {id: dep}) SET d:Service
MERGE (s)-[:DEPENDS_ON]->(d)""", {"rows": svc_rows})

print("Structured data loaded.")

Structured data loaded.


---
## Step 3: Look at the Graph

**See it visually:** open Neo4j Browser (Aura: *Open → Query*; Docker: http://localhost:7474) and run:

```cypher
MATCH (a:__Entity__)-[r]->(b:__Entity__) WHERE NOT type(r) IN ['HAS_SKILL', 'LOCATED_IN'] RETURN a, r, b
```

In [10]:
def counts():
    nodes = graph.query("""MATCH (n) RETURN [l IN labels(n) WHERE l <> '__Entity__'][0] AS label, count(*) AS nodes
                           ORDER BY nodes DESC""")
    rels = graph.query("MATCH ()-[r]->() RETURN type(r) AS relationship, count(*) AS count ORDER BY count DESC")
    return pd.DataFrame(nodes), pd.DataFrame(rels)

n, r = counts()
show("Nodes"); display(n)
show("Relationships"); display(r)
show("The org chart (who reports to whom)")
display(pd.DataFrame(graph.query("""
MATCH (e:Person)-[:REPORTS_TO]->(m:Person)
RETURN m.id AS manager, m.title AS manager_title, collect(e.id) AS direct_reports ORDER BY size(direct_reports) DESC""")))


=== Nodes ===


,label,nodes
0,Technology,31
1,Person,24
2,Service,8
3,Team,7
4,Project,4
5,Location,2



=== Relationships ===


,relationship,count
0,HAS_SKILL,53
1,MEMBER_OF,24
2,LOCATED_IN,24
3,REPORTS_TO,23
4,WORKS_ON,17
5,USES,12
6,DEPENDS_ON,9
7,MAINTAINS,8
8,LEADS,7
9,OWNS,4



=== The org chart (who reports to whom) ===


,manager,manager_title,direct_reports
0,Karan Bhatia,Engineering Manager,"[Arvind Nair, Sneha Iyer, Rahul Verma, Farhan Sheikh, Lakshmi Pillai]"
1,Arjun Mehta,Chief Executive Officer,"[Priya Sharma, Neha Kapoor, Vikram Iyer, Ravi Menon]"
2,Divya Rao,Head of Data & AI,"[Meera Joshi, Kabir Das, Ananya Singh, Rohan Gupta]"
3,Priya Sharma,Chief Technology Officer,"[Karan Bhatia, Divya Rao, Anil Kumar]"
4,Neha Kapoor,Chief Human Resources Officer,"[Pooja Reddy, Sara Thomas]"
5,Anil Kumar,Engineering Manager,"[Diya Reddy, Ishaan Menon]"
6,Vikram Iyer,Chief Financial Officer,[Nisha Jain]
7,Ravi Menon,VP Sales,[Aarav Shah]
8,Aarav Shah,Sales Manager,[Kavya Nair]


---
## Step 4: Load the Documents — GPT-4o Extracts, the Graph Links

The CSVs do not contain everything. The documents add new knowledge — for example the **incident**: what caused it, what it affected, who fixed it.

| Step | What Happens | Tool |
|---|---|---|
| 1 | PDF → Markdown → one chunk per section | `pymupdf4llm` |
| 2 | GPT-4o extracts nodes and relationships, limited to our schema | `LLMGraphTransformer` |
| 3 | Write them to Neo4j; each chunk becomes a `Document` node with `MENTIONS` links | `add_graph_documents` |

Because every node is merged on `id` (the name), **"Karan Bhatia" in the status report joins the existing Karan Bhatia node** from the CSV — the document facts and the HR facts end up on the same person.

The schema adds one new node type, **`Incident`**, with `CAUSED_BY`, `AFFECTED` and `RESOLVED` relationships.

In [12]:
!pip install pymupdf4llm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 67.9 MB/s eta 0:00:00


In [14]:
!pip install langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [15]:
import pymupdf4llm
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer

docs = []
for pdf in sorted(DATA.glob("*.pdf")):
    md = pymupdf4llm.to_markdown(str(pdf), header=False, footer=False, use_ocr=False)
    title = re.sub(r"[*#]", "", md.strip().splitlines()[0]).strip()
    for i, part in enumerate(re.split(r"\n(?=## )", md)[1:], 1):          # one chunk per "## " section
        heading = re.sub(r"[*#]", "", part.splitlines()[0]).strip()
        body = "\n".join(part.splitlines()[1:]).strip()
        docs.append(Document(page_content=f"{title} | {heading}\n{body}",
                             metadata={"id": f"{pdf.stem}#{i}", "source": pdf.name, "section": heading}))
print(f"{len(docs)} chunks from {len(list(DATA.glob('*.pdf')))} documents")
display(pd.DataFrame([{"id": d.metadata["id"], "text": d.page_content[:110].replace("\n", " | ")} for d in docs]))

NODES = ["Person", "Team", "Project", "Technology", "Service", "Location", "Incident"]
RELS = [("Person", "REPORTS_TO", "Person"), ("Person", "LEADS", "Team"), ("Person", "MEMBER_OF", "Team"),
        ("Team", "OWNS", "Project"), ("Project", "USES", "Technology"), ("Person", "WORKS_ON", "Project"),
        ("Project", "BUILDS", "Service"), ("Service", "DEPENDS_ON", "Service"), ("Team", "MAINTAINS", "Service"),
        ("Incident", "CAUSED_BY", "Service"), ("Incident", "AFFECTED", "Service"), ("Person", "RESOLVED", "Incident")]

transformer = LLMGraphTransformer(
    llm=llm, allowed_nodes=NODES, allowed_relationships=RELS, node_properties=["title", "status"], strict_mode=True,
    additional_instructions=("Use full names for people (e.g. 'Karan Bhatia'). Replace pronouns with the name they refer to. "
                             "If a person is mentioned only by a role such as 'the CTO', use that role as the id. "
                             "Use short names for teams and projects: 'Platform', 'Data & AI', 'Orion', 'Sage'. "
                             "Use the incident number as the Incident id, e.g. 'INC-2025-0917'."))
graph_docs = transformer.convert_to_graph_documents(docs)

existing = {r["id"] for r in graph.query("MATCH (e:__Entity__) RETURN e.id AS id")}
found = sorted({(n.id, n.type) for g in graph_docs for n in g.nodes})
show("Entities found in the documents")
display(pd.DataFrame([{"entity": i, "type": t, "already in graph from CSV?": "yes" if i in existing else "NEW"} for i, t in found]))

graph.add_graph_documents(graph_docs, include_source=True, baseEntityLabel=True)
show("Relationships extracted from the documents")
display(pd.DataFrame([{"chunk": g.source.metadata["id"], "from": r.source.id, "relationship": r.type, "to": r.target.id}
                      for g in graph_docs for r in g.relationships]))

10 chunks from 3 documents


,id,text
0,Engineering_Org_Update_Jul_2025#1,"Engineering Organisation Update - July 2025 | 1. Message from the CTO | Priya Sharma, our Chief Technology Offic"
1,Engineering_Org_Update_Jul_2025#2,Engineering Organisation Update - July 2025 | 2. Team Changes | Karan Bhatia continues to lead the Platform team
2,Engineering_Org_Update_Jul_2025#3,Engineering Organisation Update - July 2025 | 3. Open Positions | The Platform team is hiring two senior backend
3,Incident_Report_INC-2025-0917#1,"Incident Report INC-2025-0917 - Payments Errors | 1. What Happened | On 17 September 2025 at 10:42, a configurat"
4,Incident_Report_INC-2025-0917#2,Incident Report INC-2025-0917 - Payments Errors | 2. Resolution | Lakshmi Pillai from the Platform team rolled b
5,Incident_Report_INC-2025-0917#3,Incident Report INC-2025-0917 - Payments Errors | 3. Actions | All Redis Cluster 2 changes now need approval fro
6,Project_Orion_Status_Q3_2025#1,Project Orion - Status Report Q3 2025 | 1. Summary | Project Orion is 6 weeks behind schedule. The new target da
7,Project_Orion_Status_Q3_2025#2,Project Orion - Status Report Q3 2025 | 2. Reasons for the Delay | The migration of payments from the legacy gat
8,Project_Orion_Status_Q3_2025#3,"Project Orion - Status Report Q3 2025 | 3. Risks | The Payments API, which Orion builds, depends on the Auth Ser"
9,Project_Orion_Status_Q3_2025#4,"Project Orion - Status Report Q3 2025 | 4. Escalation | Escalations go to Karan Bhatia first, then to Priya Shar"



=== Entities found in the documents ===


,entity,type,already in graph from CSV?
0,Ananya Singh,Person,yes
1,Arjun Mehta,Person,yes
2,Arvind Nair,Person,yes
3,Atlas,Project,yes
4,Auth Service,Service,yes
5,Cto,Person,NEW
6,Data & Ai,Team,NEW
7,Divya Rao,Person,yes
8,Farhan Sheikh,Person,yes
9,Go-Live,Project,NEW



=== Relationships extracted from the documents ===


,chunk,from,relationship,to
0,Engineering_Org_Update_Jul_2025#1,Priya Sharma,REPORTS_TO,Arjun Mehta
1,Engineering_Org_Update_Jul_2025#2,Karan Bhatia,LEADS,Platform
2,Engineering_Org_Update_Jul_2025#2,Karan Bhatia,REPORTS_TO,Cto
3,Engineering_Org_Update_Jul_2025#2,Platform,OWNS,Orion
4,Engineering_Org_Update_Jul_2025#2,Divya Rao,LEADS,Data & Ai
5,Engineering_Org_Update_Jul_2025#2,Divya Rao,REPORTS_TO,Priya
6,Engineering_Org_Update_Jul_2025#2,Data & Ai,OWNS,Atlas
7,Engineering_Org_Update_Jul_2025#2,Data & Ai,OWNS,Sage
8,Engineering_Org_Update_Jul_2025#2,Farhan Sheikh,WORKS_ON,Orion
9,Engineering_Org_Update_Jul_2025#2,Farhan Sheikh,WORKS_ON,Helios


---
## Step 5: Entity Resolution — "the CTO" Is Priya Sharma

Documents often use **roles** ("the CTO", "the CEO") or **longer names** ("Project Orion", "the Platform team"). We merge each of these into the real node with APOC:

| Rule | Example |
|---|---|
| Role → the person whose **title** matches | "CTO" → Priya Sharma (title: Chief Technology Officer) |
| Long name → short name | "Project Orion" → "Orion"; "Platform team" → "Platform" |

`apoc.refactor.mergeNodes` moves **all relationships** from the duplicate to the real node.

In [16]:
ROLES = {"ceo": "Chief Executive Officer", "cto": "Chief Technology Officer", "chro": "Chief Human Resources Officer",
         "cfo": "Chief Financial Officer"}
people = {r["title"].lower(): r["id"] for r in graph.query("MATCH (p:Person) WHERE p.emp_id IS NOT NULL RETURN p.id AS id, p.title AS title")}
ids = {r["id"] for r in graph.query("MATCH (e:__Entity__) RETURN e.id AS id")}

def target_for(name):
    n = re.sub(r"^(the|our)\s+", "", name.strip(), flags=re.I)
    if n.lower() in ROLES:
        return people.get(ROLES[n.lower()].lower())
    for title, person in people.items():
        if n.lower() == title:
            return person
    short = re.sub(r"\s+team$", "", re.sub(r"^project\s+", "", n, flags=re.I), flags=re.I)
    return short if short != name and short in ids else None

merges = [(name, target_for(name)) for name in sorted(ids) if target_for(name) and target_for(name) != name]
for dup, keep in merges:
    graph.query("""MATCH (keep:__Entity__ {id: $keep}), (dup:__Entity__ {id: $dup})
                   CALL apoc.refactor.mergeNodes([keep, dup], {properties: 'discard', mergeRels: true}) YIELD node
                   RETURN node.id""", {"keep": keep, "dup": dup})
display(pd.DataFrame(merges, columns=["duplicate", "merged into"])) if merges else print("Nothing to merge.")
n, r = counts()
show("Graph after adding the documents"); display(n)

,duplicate,merged into
0,Cto,Priya Sharma
1,The Cto,Priya Sharma



=== Graph after adding the documents ===


,label,nodes
0,Technology,33
1,Person,25
2,Service,11
3,Document,10
4,Team,8
5,Project,7
6,Location,2
7,Incident,1


---
## Step 6: Vector Index in Neo4j — Watch Vector-Only RAG Fail

`Neo4jVector.from_existing_graph` embeds the text of every `Document` node and creates a vector index **inside Neo4j**.

**Question:** *"Which technologies are used by projects run by teams whose leads report to the CTO?"*

The answer is spread over the org chart (CSV) and project records — no single chunk says it. Vector search returns the most **similar** text, not the **connected** facts.

In [17]:
from langchain_neo4j import Neo4jVector

vector_store = Neo4jVector.from_existing_graph(
    embedding=embeddings, **NEO4J, index_name="org_docs", node_label="Document",
    text_node_properties=["text"], embedding_node_property="embedding")

QUESTION = "Which technologies are used by projects run by teams whose leads report to the CTO?"
hits = vector_store.similarity_search(QUESTION, k=3)
context = "\n\n".join(d.page_content for d in hits)
show("Top 3 chunks from vector search", "\n---\n".join(d.page_content.strip()[:200] for d in hits))
show("Vector-only answer", llm.invoke("Answer ONLY from the context. If it is incomplete, say what is missing.\n\n"
                                      f"Context:\n{context}\n\nQuestion: {QUESTION}").content)


=== Top 3 chunks from vector search ===
text: Engineering Organisation Update - July 2025 | 1. Message from the CTO
Priya Sharma, our Chief Technology Officer, shares the engineering structure for the second half of 2025. She continues to r
---
text: Project Orion - Status Report Q3 2025 | 4. Escalation
Escalations go to Karan Bhatia first, then to Priya Sharma, the CTO.
---
text: Engineering Organisation Update - July 2025 | 2. Team Changes
Karan Bhatia continues to lead the Platform team. He reports to the CTO and owns Project Orion. 

Divya Rao leads the Data & AI team

=== Vector-only answer ===
The context is incomplete regarding the specific technologies used by all projects run by teams whose leads report to the CTO. The only mentioned technology is for Project Sage, which uses LangChain and GPT models.


---
## Step 7: Real-World Questions Answered With Cypher

These are the use cases from Section 1.6 of the document:

| # | Question | Use Case |
|---|---|---|
| Q1 | What is Rahul Verma's full reporting chain up to the CEO? | Org chart |
| Q2 | How many people are under the CTO, by team? | Org chart, any depth |
| Q3 | Who knows **Kubernetes** and works in **Bengaluru**? | "Who knows what" |
| Q4 | Which technologies do projects under the CTO use? | Multi-hop (the Step 6 question) |
| Q5 | If **Redis Cluster 2** goes down, which services, projects and teams are affected? | Incident impact |
| Q6 | Orion is late — who works on it, and who do I escalate to? | Project escalation |
| Q7 | Who is allocated more than 100% across projects? | Resource planning |
| Q8 | What does the incident INC-2025-0917 connect to? | Document knowledge + HR knowledge |

In [18]:
QUERIES = {
"Q1. Rahul Verma's reporting chain": """
MATCH path = (:Person {id: 'Rahul Verma'})-[:REPORTS_TO*]->(top:Person)
WHERE NOT (top)-[:REPORTS_TO]->()
RETURN [p IN nodes(path) | p.id + ' (' + coalesce(p.title, '') + ')'] AS chain""",

"Q2. People under the CTO, by team": """
MATCH (cto:Person {title: 'Chief Technology Officer'})<-[:REPORTS_TO*]-(p:Person)-[:MEMBER_OF]->(t:Team)
RETURN t.id AS team, count(DISTINCT p) AS people, collect(DISTINCT p.id) AS names ORDER BY people DESC""",

"Q3. Who knows Kubernetes and works in Bengaluru?": """
MATCH (p:Person)-[:HAS_SKILL]->(:Technology {id: 'Kubernetes'}), (p)-[:LOCATED_IN]->(:Location {id: 'Bengaluru'})
OPTIONAL MATCH (p)-[w:WORKS_ON]->(pr:Project)
RETURN p.id AS person, p.title AS title, collect(pr.id + ' (' + w.role + ')') AS projects""",

"Q4. Technologies used by projects under the CTO": """
MATCH (cto:Person {title: 'Chief Technology Officer'})<-[:REPORTS_TO]-(lead:Person)-[:LEADS]->(t:Team)
      -[:OWNS]->(pr:Project)-[:USES]->(tech:Technology)
RETURN pr.id AS project, t.id AS team, lead.id AS lead, pr.status AS status, collect(tech.id) AS technologies""",

"Q5. Impact if Redis Cluster 2 goes down": """
MATCH (s:Service)-[:DEPENDS_ON*1..4]->(:Service {id: 'Redis Cluster 2'})
OPTIONAL MATCH (pr:Project)-[:BUILDS]->(s)
OPTIONAL MATCH (t:Team)-[:MAINTAINS]->(s)
RETURN s.id AS affected_service, pr.id AS built_by_project, t.id AS maintained_by""",

"Q6. Orion - who works on it, and the escalation path": """
MATCH (pr:Project {id: 'Orion'})<-[w:WORKS_ON]-(p:Person)
WITH pr, collect(p.id + ' - ' + w.role + ' ' + toString(w.allocation_pct) + '%') AS team
MATCH (pr)<-[:OWNS]-(:Team)<-[:LEADS]-(lead:Person)
MATCH path = (lead)-[:REPORTS_TO*0..]->(top:Person) WHERE NOT (top)-[:REPORTS_TO]->()
RETURN pr.status AS status, pr.target_date AS target_date, team, [p IN nodes(path) | p.id] AS escalation_path""",

"Q7. Allocated more than 100%": """
MATCH (p:Person)-[w:WORKS_ON]->(pr:Project)
WITH p, sum(w.allocation_pct) AS total, collect(pr.id + ' ' + toString(w.allocation_pct) + '%') AS projects
WHERE total > 100
RETURN p.id AS person, total AS total_allocation_pct, projects""",

"Q8. What the incident connects to (from the incident report)": """
MATCH (i:Incident)-[r]-(x:__Entity__)
RETURN i.id AS incident, type(r) AS relationship, x.id AS connected_to,
       [l IN labels(x) WHERE l <> '__Entity__'][0] AS type""",
}
for title, cypher in QUERIES.items():
    show(title, "Cypher:" + cypher)
    display(pd.DataFrame(graph.query(cypher)))


=== Q1. Rahul Verma's reporting chain ===
Cypher:
MATCH path = (:Person {id: 'Rahul Verma'})-[:REPORTS_TO*]->(top:Person)
WHERE NOT (top)-[:REPORTS_TO]->()
RETURN [p IN nodes(path) | p.id + ' (' + coalesce(p.title, '') + ')'] AS chain


,chain
0,"[Rahul Verma (Engineer), Karan Bhatia (Project Lead), Priya Sharma (Chief Technology Officer), Priya Sharma (Chief T..."
1,"[Rahul Verma (Engineer), Karan Bhatia (Project Lead), Priya Sharma (Chief Technology Officer), Arjun Mehta (CEO)]"



=== Q2. People under the CTO, by team ===
Cypher:
MATCH (cto:Person {title: 'Chief Technology Officer'})<-[:REPORTS_TO*]-(p:Person)-[:MEMBER_OF]->(t:Team)
RETURN t.id AS team, count(DISTINCT p) AS people, collect(DISTINCT p.id) AS names ORDER BY people DESC


,team,people,names
0,Platform,6,"[Karan Bhatia, Arvind Nair, Sneha Iyer, Rahul Verma, Farhan Sheikh, Lakshmi Pillai]"
1,Data & AI,5,"[Divya Rao, Meera Joshi, Kabir Das, Ananya Singh, Rohan Gupta]"
2,Mobile,3,"[Anil Kumar, Diya Reddy, Ishaan Menon]"
3,Leadership,1,[Priya Sharma]
4,Data & Ai,1,[Ananya Singh]



=== Q3. Who knows Kubernetes and works in Bengaluru? ===
Cypher:
MATCH (p:Person)-[:HAS_SKILL]->(:Technology {id: 'Kubernetes'}), (p)-[:LOCATED_IN]->(:Location {id: 'Bengaluru'})
OPTIONAL MATCH (p)-[w:WORKS_ON]->(pr:Project)
RETURN p.id AS person, p.title AS title, collect(pr.id + ' (' + w.role + ')') AS projects


...

,status,target_date,team,escalation_path
0,Q3 2025,2025-12-15,"[Karan Bhatia - Project Lead 50%, Arvind Nair - Tech Lead 100%, Sneha Iyer - Developer 100%, Rahul Verma - Developer...","[Karan Bhatia, Priya Sharma, Arjun Mehta]"
1,Q3 2025,2025-12-15,"[Karan Bhatia - Project Lead 50%, Arvind Nair - Tech Lead 100%, Sneha Iyer - Developer 100%, Rahul Verma - Developer...","[Karan Bhatia, Priya Sharma, Priya Sharma, Arjun Mehta]"
2,Q3 2025,2025-12-15,"[Karan Bhatia - Project Lead 50%, Arvind Nair - Tech Lead 100%, Sneha Iyer - Developer 100%, Rahul Verma - Developer...","[Sara Thomas, Neha Kapoor, Arjun Mehta]"



=== Q7. Allocated more than 100% ===
Cypher:
MATCH (p:Person)-[w:WORKS_ON]->(pr:Project)
WITH p, sum(w.allocation_pct) AS total, collect(pr.id + ' ' + toString(w.allocation_pct) + '%') AS projects
WHERE total > 100
RETURN p.id AS person, total AS total_allocation_pct, projects


,person,total_allocation_pct,projects
0,Farhan Sheikh,120,"[Orion 60%, Helios 60%]"



=== Q8. What the incident connects to (from the incident report) ===
Cypher:
MATCH (i:Incident)-[r]-(x:__Entity__)
RETURN i.id AS incident, type(r) AS relationship, x.id AS connected_to,
       [l IN labels(x) WHERE l <> '__Entity__'][0] AS type


,incident,relationship,connected_to,type
0,Inc-2025-0917,AFFECTED,Auth Service,Service
1,Inc-2025-0917,AFFECTED,Payments Api,Service
2,Inc-2025-0917,RESOLVED,Lakshmi Pillai,Person


**Look at Q4:** the question vector search could not answer in Step 6 — one query, four hops.
**Look at Q8:** the incident came only from a PDF, yet it is now connected to services and people that came from the CSVs.

---
## Step 8: Text-to-Cypher — Ask in English

`GraphCypherQAChain` reads the question and the graph schema, writes Cypher, runs it, and answers in a sentence.
`validate_cypher=True` fixes wrong relationship directions.

⚠️ `allow_dangerous_requests=True` is required because the LLM writes queries that run on your database. **In production, connect with a read-only Neo4j user.**

In [ ]:
from langchain_neo4j import GraphCypherQAChain

graph.refresh_schema()
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=graph, validate_cypher=True, exclude_types=["Document"],
    return_intermediate_steps=True, allow_dangerous_requests=True, top_k=30)

for q in ["Who does Karan Bhatia report to?",
          "Which people work on Project Sage and what are their roles?",
          "Which services depend on the Auth Service?",
          "Who in the Data & AI team knows LLMs?"]:
    out = cypher_chain.invoke({"query": q})
    show(q)
    print("Generated Cypher:\n" + out["intermediate_steps"][0]["query"].strip())
    print("\nAnswer:", out["result"])

---
## Step 9: Hybrid Graph + Vector RAG

Some questions need **both**: the *why* is in the documents, the *who* is in the graph.

`Neo4jVector` accepts a **`retrieval_query`** — Cypher that runs right after the vector search:

| Step | What the Query Does |
|---|---|
| 1 | Vector search finds the most similar chunks |
| 2 | Follow `MENTIONS` to the people, projects and services in those chunks |
| 3 | Collect their relationships (reports to, leads, works on, depends on, maintains …) |
| 4 | Return the chunk text **plus** those graph facts |

In [ ]:
HYBRID_QUERY = """
OPTIONAL MATCH (node)-[:MENTIONS]->(e:__Entity__)-[r]-(n:__Entity__)
WHERE NOT type(r) IN ['HAS_SKILL', 'LOCATED_IN']
WITH node, score, collect(DISTINCT startNode(r).id + ' -[' + type(r) + ']-> ' + endNode(r).id) AS facts
RETURN node.text AS text, score, {chunk: node.id, facts: facts} AS metadata
"""
hybrid_store = Neo4jVector.from_existing_index(embedding=embeddings, **NEO4J, index_name="org_docs",
                                               retrieval_query=HYBRID_QUERY)

def graph_rag(question, k=2):
    hits = hybrid_store.similarity_search(question, k=k)
    texts = [f"[{d.metadata['chunk']}] {d.page_content}" for d in hits]
    facts = sorted({f for d in hits for f in d.metadata["facts"] if f})
    prompt = ("Answer using the document text and the graph facts. Name the people and the path you followed.\n\n"
              "Document text:\n" + "\n\n".join(texts) + "\n\nGraph facts:\n" + "\n".join(facts) +
              f"\n\nQuestion: {question}")
    return texts, facts, llm.invoke(prompt).content

for q in ["Why is Project Orion delayed, and who should I escalate to?",
          "What happened in the September payments incident, who fixed it, and which team owns the fix?"]:
    texts, facts, answer = graph_rag(q)
    print("\n" + "=" * 100 + f"\nQUESTION: {q}")
    show("Document text (vector search)", "\n".join(t[:300] for t in texts))
    show(f"Graph facts ({len(facts)})", "\n".join(facts[:15]) + ("\n..." if len(facts) > 15 else ""))
    show("Answer", answer)

---
## Summary

| Step | Tool | What It Did |
|---|---|---|
| 2 | Cypher `UNWIND` | Loaded 24 people, 7 teams, 4 projects, 8 services — exact, by ID |
| 4 | `LLMGraphTransformer` + `add_graph_documents` | Added document knowledge (the incident, the delay) onto the **same** nodes |
| 5 | `apoc.refactor.mergeNodes` | "CTO" → Priya Sharma; "Project Orion" → Orion |
| 6 | `Neo4jVector` | Vector search inside Neo4j — fails on multi-hop questions |
| 7 | Cypher | Org chart, who-knows-what, incident impact, escalation, over-allocation |
| 8 | `GraphCypherQAChain` | English → Cypher → answer |
| 9 | `Neo4jVector` + `retrieval_query` | Why (documents) + who (graph) in one answer |

> **Structured data gives the graph its skeleton. Documents add the story.** Graph RAG connects both.